# Multi-location energy planning for Houston, San Diego, and New York

Decide where to build generating and electrolysis capacity, and route energy from the two production sites to New York. The legacy rail-only case delivers hydrogen to a fuel cell in New York. A second solve enables the legacy notebook's grid option to compare hydrogen rail with direct power transmission.

## Location and link inputs

The profiles use four six-hour blocks in UTC. Solar availability is derived from the legacy NSRDB series for Houston and San Diego on July 15, 2019. The New York demand shape comes from July 15, 2018, the load year included in the legacy case. The demand is scaled to a 35 MW peak. Routes use the legacy distances of 2,366 and 2,620 miles and the original nominal efficiencies of 97% for hydrogen and 99.9% for electricity. Costs are illustrative.

In [1]:
from math import isclose

import pandas as pd
from energia import Location, Model, Periods, Process, Resource, Transport
from gana import inf

pv_profile = {'houston': [0.0079, 0.0, 0.1141, 0.1402], 'san_diego': [0.154, 0.0, 0.4087, 0.935]}
demand = [33.75, 27.67, 30.85, 35.0]
distances = {"houston": 2366, "san_diego": 2620}
block_hours = 6

## Make the spatial formulation

Hydrogen and power have separate mass balances. The rail service ships tonnes of hydrogen and the grid service ships electricity. A fuel cell produces electricity from rail deliveries after the link losses.

In [2]:
def plan_network(enable_grid):
    m = Model("rail_and_grid" if enable_grid else "rail_only")
    m.block = Periods()
    m.day = len(demand) * m.block
    m.declare(Location, ["houston", "san_diego", "new_york"])
    m.declare(Resource, ["sun", "gas", "power", "hydrogen"])
    for site in (m.houston, m.san_diego, m.new_york):
        _ = m.power.consume(site, m.block) == 0
        _ = m.hydrogen.consume(site, m.block) == 0
    served = m.power.release(m.new_york, m.block)
    _ = served == demand
    cost = 0

    for site_name, miles in distances.items():
        site = getattr(m, site_name)
        _ = m.sun.consume(site, m.block) >= 0
        _ = m.gas.consume(site, m.block) >= 0
        plants = (
            ("pv", m.power == -m.sun, pv_profile[site_name], 85 if site_name == "houston" else 72, 0),
            ("gas", m.power == -0.108 * m.gas, 1, 80, 55 if site_name == "houston" else 70),
            ("electrolysis", m.hydrogen == -52 * m.power, 1, 100, 0),
        )
        for name, conversion, availability, capex, opex in plants:
            plant = Process(conversion)
            setattr(m, f"{site_name}_{name}", plant)
            capacity = plant.capacity(site, m.day)
            dispatch = plant.operate(site, m.block)
            _ = capacity <= 200
            _ = dispatch <= availability
            plant.locate(site)
            cost += capex * capacity.V()[0]
            if opex:
                cost += block_hours * opex * sum(dispatch.V())
        m.Link(site, m.new_york, dist=miles)

    m.fuel_cell = Process(m.power == -0.05 * m.hydrogen)
    cell_capacity = m.fuel_cell.capacity(m.new_york, m.day)
    _ = cell_capacity <= 40
    _ = m.fuel_cell.operate(m.new_york, m.block) <= 1
    m.fuel_cell.locate(m.new_york)
    cost += 120 * cell_capacity.V()[0]

    m.rail = Transport()
    _ = m.rail(m.hydrogen) == 0.97
    if enable_grid:
        m.grid = Transport()
        _ = m.grid(m.power) == 0.999

    shipments = {}
    for site_name, miles in distances.items():
        link = getattr(m, site_name) - m.new_york
        services = [(m.rail, 10, 0.001667)]
        if enable_grid:
            services.append((m.grid, 16, 0.0005))
        for service, limit, rate in services:
            _ = service.capacity(link, m.day) == limit
            dispatch = service.operate(link, m.block)
            _ = dispatch <= 1
            service.locate(link)
            cost += block_hours * miles * rate * sum(dispatch.V())
            shipments[f"{site_name} {service.name}"] = dispatch

    m.program.delivered_cost = inf(cost)
    m.program.opt()
    assert all(isclose(a, b, abs_tol=1e-6) for a, b in zip(served.output(aslist=True), demand, strict=True))
    flows = {name: block_hours * sum(sample.output(aslist=True)) for name, sample in shipments.items()}
    return {"Daily cost": m.program.active_objective.X, **flows}

## Compare transport options

Rail shipments are in tonnes of hydrogen. Grid flows are in MWh. The option to use direct transmission can reduce cost because the model may still use rail whenever it is cheaper or needed to meet demand.

In [3]:
rail = plan_network(enable_grid=False)
combined = plan_network(enable_grid=True)
assert rail["Daily cost"] >= combined["Daily cost"] - 1e-6
assert rail["houston rail"] + rail["san_diego rail"] > 0
assert combined["houston grid"] + combined["san_diego grid"] > 0
pd.DataFrame([rail, combined], index=["Rail only", "Rail and grid"]).fillna(0).round(2)

,Daily cost,houston rail,san_diego rail,houston grid,san_diego grid
Rail only,67284.21,14.94,23.24,0.00,0.00
Rail and grid,30409.82,0.00,10.68,191.94,358.02
